In [3]:
TEMP_DIR = "../data/tmp"
CHANNELS_YAML = '../data/channels.yaml'

import yaml

with open(CHANNELS_YAML, 'r') as stream:
    try:
        channels = yaml.safe_load(stream)
    except yaml.YAMLError as exc:
        channels = {}
        print(exc)
        
from pathlib import Path

Path(TEMP_DIR).mkdir(parents=True, exist_ok=True)


keys = channels.keys()

handles = [channels.get(key) for key in keys]
handles = [el for sub in handles for el in sub]

handles

['johnnyharris',
 'PolyMatter',
 'neoexplains',
 'SearchParty',
 'Faultlinevideos',
 'RealLifeLore',
 'AtlasPro1',
 'WendoverProductions',
 'halfasinteresting',
 'SamONellaAcademy',
 'CasuallyExplained',
 'CGPGrey',
 'InternetHistorian',
 'TierZoo',
 'billwurtz',
 'ApertureThinking',
 'Exurb1a',
 'theschooloflifetv',
 'LEMMiNO',
 'FredrikKnudsen',
 'blameitonjorge',
 'BarelySociable',
 'Nexpo',
 'RealEngineering',
 'ColdFusion',
 'MustardChannel',
 'TheB1M',
 'Kurzgesagt',
 'BobbyBroccoli',
 'TomScottGo',
 'Vsauce',
 'veritasium',
 'VICE',
 'Vox',
 'wsj',
 'JacobGeller',
 'BigJoel',
 'KnowingBetter',
 'hbomberguy',
 'Shaun_vids']

In [ ]:
data_dir = "../data/tmp"

import os

# List all JSON files in the data directory
json_files = [f for f in os.listdir(data_dir) if f.endswith('.json')]

from tqdm.notebook import tqdm
import polars as pl

keep_cols = [
    'id', 'title', 'media_type', 'duration', 
    'view_count',
    'uploader_id', 'channel_id'
]

# Create a Polars DataFrame from the JSON files
frames = [pl.read_json(os.path.join(data_dir, f)).select(keep_cols) 
          for f in tqdm(json_files)]

df = pl.concat(frames)

df

  0%|          | 0/10155 [00:00<?, ?it/s]

10155

In [ ]:
all_columns = sorted(set().union(*[set(df.columns) for df in frames]))

aligned_frames = []
pbar = tqdm(total=len(frames), desc="Aligning columns")

for df in frames:
    missing_cols = [col for col in all_columns if col not in df.columns]
    # Add missing columns with nulls in a single call
    if missing_cols:
        df = df.with_columns([pl.lit(None).alias(col) for col in missing_cols])
    # Select and reorder columns once
    df = df.select(all_columns)
    aligned_frames.append(df)
    pbar.update(1)

pbar.close()

df = pl.concat(aligned_frames, how='vertical_relaxed', rechunk=True)
df.shape

In [ ]:
def filter_handles(df, handles):
    handle_list = [f'@{handle}' for handle in handles]
    df = df.filter(pl.col('uploader_id').is_in(handle_list))
    return df

def rename_columns(df):
    df = df.rename({
        'uploader_id': 'channel_handle',
        'uploader': 'channel_name',
        'title': 'video_title',
        'description': 'video_description',
        'upload_date': 'date_uploaded',
        'id': 'video_id',
    })
    return df

def reorder_columns(df):
    order = [
        'video_id', 'video_title', 'video_description', 'media_type', 
        'duration', 'fps', 'height', 'width', 'date_uploaded', 'timestamp',
        'filesize_approx', 'chapters','thumbnail', 'categories', 'tags',
        'view_count', 'like_count', 'comment_count', 'automatic_captions', 
        'subtitles', 'heatmap', 'channel_name', 'channel_handle', 'channel_id'
        ]
    df = df.select(order)
    return df

def transform_data(df):
    # df = filter_handles(df, handles)
    df = rename_columns(df)
    df = reorder_columns(df)
    return df

transformed_df = transform_data(df)

transformed_df.shape

In [ ]:
automatic_captions = (
    transformed_df
    .select('automatic_captions')
    .unnest('automatic_captions')
)

import polars.selectors as cs

# select cols start with en
captions_en = automatic_captions.select(cs.starts_with('en'))
# fill nulls with other cells
captions_en = (
    captions_en
    .with_columns(
        pl.struct(captions_en.columns)
        .map_elements(
            lambda row: next(
                (val for val in row.values() if val is not None), None)
            ).alias("captions")
        )
)

caption_urls = []
for row in captions_en.iter_rows(named=True):
    caption_url = None
    caption_df = pl.DataFrame(row['captions'])
    if 'ext' in caption_df.columns:
        caption_url = caption_df.filter(pl.col('ext') == 'srt')['url'].first()
    caption_urls.append(caption_url)

data_df = transformed_df.with_columns(
    pl.Series(caption_urls).alias('automatic_captions')
)

data_df

In [ ]:
import polars as pl
import polars.selectors as cs

def extract_srt_url(entry):
    if entry is None:
        return None
    df = pl.DataFrame([entry]) if isinstance(entry, dict) else pl.DataFrame(entry)
    if 'ext' in df.columns and 'url' in df.columns:
        filtered = df.filter(pl.col('ext') == 'srt')
        if filtered.height == 1:
            return filtered['url'].item()
        elif filtered.height > 1:
            # If there are multiple, return the first one
            return filtered['url'].first()
        else:
            None
    return None

def extract_caption_url(df: pl.DataFrame, column_name: str) -> list:
    unnested = df.select(column_name).unnest(column_name)
    english_fields = (
        unnested
        .select(cs.starts_with("en"))
        .select(
            pl.struct(cs.all()).map_elements(
                lambda row: next((v for v in row.values() if v is not None), None)
            ).alias("entry")
        )
    )
    return [extract_srt_url(row) for row in english_fields["entry"].to_list()]

# Apply to automatic_captions
caption_urls = extract_caption_url(transformed_df, "automatic_captions")

# Apply to subtitles
subtitle_urls = extract_caption_url(transformed_df, "subtitles")

# Combine both into the original DataFrame
data_df = transformed_df.with_columns([
    pl.Series(name="automatic_captions", values=caption_urls),
    pl.Series(name="subtitles", values=subtitle_urls)
])

data_df.shape

In [ ]:
# write to file
output_file = '../data/video_data.parquet'
data_df.write_parquet(output_file)